In [ ]:
import os
import glob
import random
import pandas as pd
import numpy as np
import torch  
import matplotlib.pyplot as plt
import joblib
from collections import defaultdict
from enum import Enum
import seaborn as sns
from pathlib import Path
from numba import njit, float32, int64, types
from numba.typed import Dict
from tqdm import tqdm
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [ ]:
# ==========================================
# 1. 설정 (Configuration)
# ==========================================
# 실제 데이터가 있는 경로로 수정하세요
BASE_PATH = "C:/Users/user/Desktop/IDS_masters/Car_Hacking_Challenge_Dataset_rev20Mar2021/0_Preliminary/1_Submission"
PT_SAVE_PATH = "C:/Users/user/Desktop/IDS_masters/dataset/training_dataset4.pt"
CSV_SAVE_PATH = "C:/Users/user/Desktop/IDS_masters/dataset/training_dataset4.csv"
WINDOW_SIZE = 64
STRIDE = 32  # 50% Overlap

# 공격 라벨 정의
ATTACK_LABELS = {
    "Normal": 0,
    "Flooding": 1,
    "Fuzzing": 2,
    "Replay": 3,
    "Spoofing": 4
}
LABEL_MAP = {
    "Normal": 0,
    "Flooding": 1,   # 원본 명칭
    "DoS": 1,        # 혹시 나중에 DoS라는 문자열도 들어오면 같이 1로 처리
    "Fuzzing": 2,
    "Replay": 3,
    "Spoofing": 4,
}

FEATURE_NAMES = [
    "1. ID IAT", "2. Dos_ID", 
    "3. Entropy", "4.Complexity", "5. Hamming_rate", "6. Frequency", "7.Markov pattern"
]


In [ ]:
def build_transition_ref_map(can_ids):
    """
    정상 패킷들의 ID 시퀀스를 분석하여 전이 확률 맵을 생성합니다.
    """
    # 1. 빈도수 카운트를 위한 임시 딕셔너리
    counts = {}
    total_transitions = {}

    prev_id = -1
    for cid in can_ids:
        cid = int(cid)
        if prev_id != -1:
            key = (prev_id << 32) | cid
            counts[key] = counts.get(key, 0) + 1
            total_transitions[prev_id] = total_transitions.get(prev_id, 0) + 1
        prev_id = cid

    # 2. Numba 호환 Dict 생성 (Key: int64, Value: float64)
    ref_map = Dict.empty(key_type=types.int64, value_type=types.float64)

    # 3. 빈도수를 확률로 변환
    for key, count in counts.items():
        prev_id_part = key >> 32
        # P(현재ID | 이전ID) 계산
        prob = count / total_transitions[prev_id_part]
        ref_map[key] = prob

    return ref_map

In [ ]:
# ==============================
# 0. Enum 정의 (선택사항)
# ==============================
class CycleBin(Enum):
    FAST = 0
    MID = 1
    SLOW = 2
    EVENT = 3


class FreqBin(Enum):
    HIGH = 0
    MID = 1
    LOW = 2


# ==============================
# 1. 정상 패킷 기반 ID 통계 계산
# ==============================
def compute_id_stats(timestamps: np.ndarray,
                     can_ids: np.ndarray,
                     labels: np.ndarray,
                     normal_label: int = 0):
    """
    각 ID별로:
      - freq_count: 정상 패킷 수
      - iat_sum, iat_sum_sq, iat_count: IAT 통계
    를 계산해 반환한다.
    """
    # id별 통계를 dict로 저장
    stats = defaultdict(lambda: {
        "iat_sum": 0.0,
        "iat_sum_sq": 0.0,
        "iat_count": 0,
        "freq_count": 0,
        "last_ts": None
    })

    n = len(timestamps)
    for i in range(n):
        if labels[i] != normal_label:
            continue  # 공격 패킷은 통계에서 제외

        ts = float(timestamps[i])
        cid = int(can_ids[i])

        s = stats[cid]
        s["freq_count"] += 1

        if s["last_ts"] is not None:
            iat = ts - s["last_ts"]
            if iat > 0:
                s["iat_sum"] += iat
                s["iat_sum_sq"] += iat * iat
                s["iat_count"] += 1

        s["last_ts"] = ts

    return stats


# ==============================
# 2. IAT 기반 CycleBin 할당
#    - 퍼센타일 기반 Fast/Mid/Slow
#    - IAT 샘플 부족/변동성 크면 Event
# ==============================
def assign_cycle_bins(stats: dict,
                      min_iat_samples: int = 5,
                      cv_threshold: float = 0.3,
                      fast_quantile: float = 0.2,
                      mid_quantile: float = 0.6):
    """
    각 ID의 평균 IAT와 변동(CV)을 이용해서
    - 주기적인 ID들만 Fast/Mid/Slow로,
    - 나머지는 Event로 분류한다.
    """
    mean_iats = []
    id_to_mean_iat = {}

    # 1) 주기적인 ID 후보 모으기
    for cid, s in stats.items():
        if s["iat_count"] < min_iat_samples:
            # IAT 샘플 부족 → Event 후보
            continue

        mean = s["iat_sum"] / max(s["iat_count"], 1)
        var = (s["iat_sum_sq"] / max(s["iat_count"], 1)) - mean * mean
        if var < 0:
            var = 0.0
        std = np.sqrt(var)
        cv = std / mean if mean > 0 else 0.0

        if cv > cv_threshold:
            # 주기가 너무 불안정 → Event 후보
            continue

        mean_iats.append(mean)
        id_to_mean_iat[cid] = mean

    cycle_bins = {}

    # 주기적이라고 볼 ID가 거의 없으면 전부 Event로
    if len(mean_iats) < 3:
        for cid in stats.keys():
            cycle_bins[cid] = CycleBin.EVENT
        return cycle_bins

    # 2) 퍼센타일 기반 threshold 계산
    mean_iats_sorted = np.sort(np.array(mean_iats))

    def get_quantile(q: float) -> float:
        # q in [0,1]
        pos = q * (len(mean_iats_sorted) - 1)
        idx = int(pos)
        return float(mean_iats_sorted[idx])

    fast_thr = get_quantile(fast_quantile)  # 이 값 이하 → Fast
    mid_thr = get_quantile(mid_quantile)    # fast_thr~mid_thr → Mid, 나머지 → Slow

    # 3) 각 ID에 대해 cycle_bin 할당
    for cid, s in stats.items():
        if cid not in id_to_mean_iat:
            # 주기 후보가 아니었던 경우 → Event
            cycle_bins[cid] = CycleBin.EVENT
            continue

        mean_iat = id_to_mean_iat[cid]
        if mean_iat <= fast_thr:
            cycle_bins[cid] = CycleBin.FAST
        elif mean_iat <= mid_thr:
            cycle_bins[cid] = CycleBin.MID
        else:
            cycle_bins[cid] = CycleBin.SLOW

    return cycle_bins


# ==============================
# 3. 빈도 기반 FreqBin 할당
#    - 퍼센타일 기반 High/Mid/Low
# ==============================
def assign_freq_bins(stats: dict,
                     high_quantile: float = 0.9,
                     mid_quantile: float = 0.6):
    """
    각 ID의 freq_count를 이용해서
    - 상위 qHigh 이상 → High
    - 그 아래 qMid 이상 → Mid
    - 나머지 → Low
    """
    freqs = [s["freq_count"] for s in stats.values()]

    freq_bins = {}

    if len(freqs) < 3:
        # ID가 너무 적으면 그냥 다 Mid로 처리
        for cid in stats.keys():
            freq_bins[cid] = FreqBin.MID
        return freq_bins

    freqs_sorted = np.sort(np.array(freqs))

    def get_quantile_val(q: float) -> int:
        pos = q * (len(freqs_sorted) - 1)
        idx = int(pos)
        return int(freqs_sorted[idx])

    mid_thr = get_quantile_val(mid_quantile)
    high_thr = get_quantile_val(high_quantile)

    for cid, s in stats.items():
        f = s["freq_count"]
        if f >= high_thr:
            freq_bins[cid] = FreqBin.HIGH
        elif f >= mid_thr:
            freq_bins[cid] = FreqBin.MID
        else:
            freq_bins[cid] = FreqBin.LOW

    return freq_bins


# ==============================
# 4. CycleBin + FreqBin → state_code 매핑
# ==============================
def encode_state(cycle_bin: CycleBin, freq_bin: FreqBin) -> int:
    """
    표에서 정의한 대로 state code를 부여:
      Fast-High → 0
      Fast-Mid  → 1
      Fast-Low  → 2
      Mid-High  → 3
      Mid-Mid   → 4
      Mid-Low   → 5
      Slow-High → 6
      Slow-Mid  → 7
      Slow-Low  → 8
      Event-*   → 9
    """
    if cycle_bin == CycleBin.EVENT:
        return 9  # Event-Any

    # base: Fast=0~2, Mid=3~5, Slow=6~8
    if cycle_bin == CycleBin.FAST:
        base = 0
    elif cycle_bin == CycleBin.MID:
        base = 3
    else:  # SLOW
        base = 6

    if freq_bin == FreqBin.HIGH:
        freq_idx = 0
    elif freq_bin == FreqBin.MID:
        freq_idx = 1
    else:  # LOW
        freq_idx = 2

    return base + freq_idx


# ==============================
# 5. 최종: ID → state_code 맵 생성
# ==============================
def build_id_state_map(timestamps: np.ndarray,
                       can_ids: np.ndarray,
                       labels: np.ndarray):
    """
    전체 normal 구간으로부터
    - ID별 IAT, freq 통계 계산
    - CycleBin, FreqBin 할당
    - ID -> state_code (0~9) 딕셔너리 생성
    """
    # 1) 정상 패킷 기반 통계
    stats = compute_id_stats(timestamps, can_ids, labels)

    # 2) cycle, freq bin 할당
    cycle_bins = assign_cycle_bins(stats)
    freq_bins = assign_freq_bins(stats)

    # 3) ID → state_code 매핑
    id_to_state = {}
    for cid in stats.keys():
        cbin = cycle_bins.get(cid, CycleBin.EVENT)
        fbin = freq_bins.get(cid, FreqBin.MID)
        state_code = encode_state(cbin, fbin)
        id_to_state[cid] = state_code

    return id_to_state


# ==============================
# 6. 전체 패킷을 state 시퀀스로 인코딩
# ==============================
def encode_sequence_with_states(can_ids: np.ndarray,
                                id_to_state: dict,
                                default_state: int = 9):
    """
    전체 CAN ID 시퀀스를 state_code 시퀀스로 변환.
    normal에서 한 번도 등장하지 않았던 ID는 default_state (기본=9, Event 취급).
    """
    states = np.empty_like(can_ids, dtype=np.int32)
    for i, cid in enumerate(can_ids):
        cid_int = int(cid)
        states[i] = id_to_state.get(cid_int, default_state)
    return states


In [ ]:
# 7. Markov 자원 한 번에 만드는 헬퍼
# ==============================
def build_markov_resources(timestamps: np.ndarray,
                           can_ids: np.ndarray,
                           labels_int: np.ndarray):
    """
    - ID -> state_code 맵
    - 전체 state 시퀀스
    - normal state 시퀀스 기반 마르코프 전이 맵
    을 한 번에 생성.
    """
    id_to_state = build_id_state_map(timestamps, can_ids, labels_int)
    state_sequences = encode_sequence_with_states(can_ids, id_to_state)
    transition_ref_map = build_transition_ref_map(
        state_sequences[labels_int == 0]
    )
    return id_to_state, state_sequences, transition_ref_map


In [ ]:
# %%
@njit
def popcount64(x):
    # x: uint8 -> 0~255
    c = 0
    v = x
    while v:
        v &= v - np.uint64(1)
        c += 1
    return c

@njit
def pack_payload_u64(row):
    v = np.uint64(0)
    for i in range(8):
        v |= np.uint64(row[i]) << (i * 8)
    return v

@njit(fastmath=True)
def calculate_features_numba(timestamps, can_ids, state_codes,payloads, transition_ref_map):
    n = len(timestamps)
    features = np.zeros((n, 7), dtype=np.float32)
    
    
    last_time_map = Dict.empty(key_type=types.int64, value_type=types.float64) # 이전 패킷 시간
    #last_iat_map = Dict.empty(key_type=types.int64, value_type=types.float64) #이전 IAT
    last_payload_map = Dict.empty(key_type=types.int64, value_type=types.uint64) #이전 ID Payload
    last_id_map = Dict.empty(key_type=types.int64, value_type=types.int64) # 윈도우 내 id 빈도수
    

    # iat_history_map = Dict.empty(key_type=types.int64, value_type=types.float64[:])
    # ent_history_map = Dict.empty(key_type=types.int64, value_type=types.float64[:])
    # count_map = Dict.empty(key_type=types.int64, value_type=types.int64)
    # global_count_map = Dict.empty(key_type=types.int64, value_type=types.int64)
    prev_state = np.int64(-1) 
    prev_global_time = timestamps[0]
    eps = 1e-9
    # --- Markov 관련 변수 ---
    prev_id = np.int64(-1) # 이전 패킷의 ID 저장용
    eps = 1e-9
    min_prob = 1e-5 # 기준 행렬에 없는 전이가 나타날 경우 부여할 최소 확률 (Penalizing)

    prev_global_time = timestamps[0]

    for i in range(n):
        if (i % 64) == 0:
            last_id_map.clear()
        ts = timestamps[i]
        cid = can_ids[i]
        sc = state_codes[i]

        if np.isnan(ts): ts = prev_global_time

        else: prev_global_time = ts

        row = payloads[i]
        
    
        
        # 1. [Index 1] ID IAT
        if cid in last_time_map: 
            id_iat = max(0.0, ts - last_time_map[cid])
        else: 
            id_iat = 0.0
        features[i, 0] = float32(np.log1p(id_iat * 1000.0) / 7.0) 
        
        last_time_map[cid] = ts # 다음 계산을 위해 업데이트
        
        # --- 페이로드 관련 공통 준비 (Entropy, Mean, Std용) ---
        counts = np.zeros(256, dtype=np.int32)
        row = payloads[i]
        s = 0.0
        for b_idx in range(8):
            val = row[b_idx]
            counts[val] += 1
            s += val

        # 2.[index 2] : ID가 0x000인지 여부(Dos 구분에 매우 중요!)
        is_zero_id = 1.0 if cid == 0 else 0.0
        features[i, 1] = float32(is_zero_id)
        

        # 3. [Index 2] Entropy
        ent = 0.0
        for c in counts:
            if c > 0:
                p = c / 8.0
                ent -= p * np.log(p)
        features[i, 2] = float32(ent / 2.1)

       
        
        # # 4. [Index 3] Jitter
        # if cid in last_iat_map: 
        #     jitter = abs(id_iat - last_iat_map[cid])
        # else: 
        #     jitter = 0.0
        # features[i, 2] = float32(min(jitter, 0.05) / 0.05)
        # last_iat_map[cid] = id_iat # 업데이트

        # 5. ID Hamming
        cur_bytes = pack_payload_u64(row)

        if cid in last_payload_map:
            diff = cur_bytes ^ last_payload_map[cid]
            id_ham = popcount64(diff)

        else:
            id_ham = 0

        last_payload_map[cid] = cur_bytes
        #features[i,3] = float32(id_ham / 64)

       # 4. complexity
        compelxity = ent*id_ham
        if compelxity == 0:
            features[i,3] = float32(0.0)
        else:
            features[i,3] = float32(np.log1p(compelxity))

        

        #5. hamming rate
        ham_rate = id_ham / (id_iat + eps)
        if ham_rate == 0:
            features[i,4] = float32(0.0)
        else:
            features[i,4] = float32(np.log1p(ham_rate))
             
        # 6. Frequency
        if cid in last_id_map:
            cnt = last_id_map[cid]+1

        else:
            cnt = 1

        last_id_map[cid] = int64(cnt)
        features[i,5] = float32(cnt / 64)
        
        # --- 7. Markov Score 계산 (전달받은 map 사용) ---
        if prev_state != -1:
            # 일반화를 위해 추상화된 상태(0~9) 간의 전이 확인
            transition_key = (prev_state << 32) | sc
            if transition_key in transition_ref_map:
                prob = transition_ref_map[transition_key]
            else:
                prob = min_prob
            features[i, 6] = float32(-np.log(prob) / 11.5)
        else:
            features[i, 6] = float32(0.0)

        prev_state = sc

    return features

In [20]:
# ==========================================
# 2. 헬퍼 함수 (ID 파싱, Payload 파싱)
# ==========================================
def parse_id(id_val):
    if isinstance(id_val, str):
        try:
            return int(id_val, 16)
        except:
            return 0
    return int(id_val)

def parse_payload_str(s, max_len=8):
    """'00 00 A1 ...' 형태의 문자열을 길이 8의 리스트로 변환"""
    parts = str(s).split()
    vals = []
    for p in parts:
        if p != "":
            try:
                vals.append(int(p, 16))
            except:
                pass
    
    if len(vals) < max_len:
        vals += [0] * (max_len - len(vals))
    return vals[:max_len]

In [ ]:
# 2. 데이터셋 생성 함수 수정
def make_dataset_from_csv(
    BASE_PATH: str,
    window_size: int = 64,
    stride: int = 32
):
    # 스크린샷 기준 컬럼: timestamp, can_id, dlc, payload, label1, label2
    # header가 없다면 None으로 읽음
    df = pd.read_csv(BASE_PATH, header=None)

    # [수정] 라벨이 포함된 4, 5번 인덱스 컬럼(Normal/Attack, Replay)은 사용하지 않음
    # 필요한 0~3번 컬럼만 슬라이싱
    df = df.iloc[:, :4]
    df.columns = ["timestamp", "can_id", "dlc", "payload"]

    print(f"[INFO] 데이터 로드 완료. 총 패킷 수: {len(df)}")
    timestamps = df["timestamp"].astype(np.float64).to_numpy()
    can_ids = df["can_id"].apply(parse_id).astype(np.int64).to_numpy()
    dlcs = df["dlc"].astype(np.int64).to_numpy()
    payload_array = np.vstack(df["payload"].apply(parse_payload_str).values).astype(np.uint8)

    # 모든 패킷을 Normal로 가정 (따로 라벨이 없기 때문)
    dummy_labels = np.zeros_like(can_ids, dtype=np.int64)

    # Markov 자원 생성
    _, state_codes, transition_ref_map = build_markov_resources(
        timestamps, can_ids, dummy_labels
    )
    # 3. 전체 데이터에 대해 피처 미리 계산 (Numba 가속 함수 사용)
    # calculate_features_numba는 (N, 9) 형태의 행렬을 반환함
    print("[INFO] 피처 계산 중...")
    all_features = calculate_features_numba(
        timestamps, can_ids, state_codes, payload_array, transition_ref_map
    )
    # 4. 윈도우 슬라이싱 (9, 64) 형태로 변환
    N = len(all_features)
    windows_features = []

    for start in range(0, N - window_size + 1, stride):
        end = start + window_size
        # (window_size, 9) -> (9, window_size)로 전치하여 모델 입력 규격 맞춤
        win_feat = all_features[start:end].T
        windows_features.append(win_feat)

    if not windows_features:
        raise RuntimeError("윈도우를 생성할 수 없습니다. 데이터가 WINDOW_SIZE보다 적습니다.")

    X = np.stack(windows_features, axis=0)  # (num_windows, 9, 64)
    return X

In [22]:
# ==========================================
# 4. 윈도우 자르기 (Sequence Labeling 수정 완료)
# ==========================================
def slice_windows(features, packet_labels, window_size, stride):
    """
    packet_labels: 이미 0, 1, 2, 3, 4가 마킹된 패킷 라벨 배열
    반환값: 
      - X: (N, 9, 64)
      - y: (N, 64)  <-- 벡터 형태
    """
    n_samples = len(features)
    if n_samples < window_size: return None, None
    
    n_windows = (n_samples - window_size) // stride + 1
    
    X_list = []
    y_list = []
  
    for i in range(n_windows):
        start = i * stride
        end = start + window_size
        
        # 1. Feature 자르기
        win_feat = features[start:end]
        
        # 2. Label 자르기 (이미 ID가 부여되어 있으므로 그대로 자름)
        # [중요] np.where 등을 쓰지 않고 그대로 가져옵니다.
        win_y = packet_labels[start:end]
        
        # 길이 체크 (마지막 짜투리 방지)
        if len(win_y) == window_size:
            X_list.append(win_feat.T) # (9, 64)로 전치
            y_list.append(win_y)      # (64,) 벡터
        
    return np.array(X_list, dtype=np.float32), np.array(y_list, dtype=np.int64)

In [ ]:

def make_dataset_with_labels(csv_path, window_size=64, stride=32):
    # 1. 데이터 로드 (모든 컬럼 읽기)
    # 스크린샷 기준: 0:ts, 1:id, 2:dlc, 3:payload, 4:label1(Normal/Attack), 5:label2(공격유형)
    df = pd.read_csv(csv_path, header=None)
    
    # 분석용 라벨 추출 (5번 컬럼: Replay, DoS 등 상세 라벨)
    raw_labels = df.iloc[:, 5].values 
    labels_int = np.vectorize(LABEL_MAP.get)(raw_labels).astype(np.int64)
    # 피처 추출용 데이터 정리
    timestamps = df.iloc[:, 0].astype(np.float64).to_numpy()
    can_ids = df.iloc[:, 1].apply(parse_id).astype(np.int64).to_numpy()

    payload_array = np.vstack(df.iloc[:, 3].apply(parse_payload_str).values).astype(np.uint8)
    # Markov 자원 생성
    _, state_codes, transition_ref_map = build_markov_resources(
        timestamps, can_ids, labels_int
    )

    # 2. 피처 계산 (Numba)
    all_features = calculate_features_numba(
        timestamps, can_ids, state_codes, payload_array, transition_ref_map
    )

    X_list = []
    label_list = []

    # 3. 윈도우 슬라이싱 및 라벨 결정
    for start in range(0, len(all_features) - window_size + 1, stride):
        end = start + window_size
        
        # 모델용 피처 (9, 64)
        X_list.append(all_features[start:end].T)
        
        # [중요] CSV 확인용 라벨 결정: 윈도우 내의 마지막 패킷 라벨 혹은 대표 라벨 사용
        # 여기서는 사람이 읽기 편하도록 해당 윈도우의 '마지막 패킷 라벨'을 할당합니다.
        label_list.append(raw_labels[start:end])

    return np.array(X_list, dtype=np.float32), np.array(label_list, dtype=np.int64)

In [ ]:
def main():
    # 1. CSV 파일 목록
    csv_files = glob.glob(os.path.join(BASE_PATH, "*.csv"))
    if not csv_files:
        print(f"[ERROR] 해당 경로에 CSV 파일이 없습니다: {BASE_PATH}")
        return

    print(f"[INFO] 발견된 파일: {len(csv_files)}개")
    for f in csv_files:
        print("   -", os.path.basename(f))

    # 2. CSV 통합
    df_list = []
    for file in csv_files:
        print(f"[READING] {os.path.basename(file)} 읽는 중...")
        temp_df = pd.read_csv(file, header=0)
        df_list.append(temp_df)

    full_df = pd.concat(df_list, axis=0, ignore_index=True)
    print(f"[INFO] 통합 완료. 총 패킷 수: {len(full_df)}")

    # 3. 라벨 정리 (Flooding → DoS 이름 통일)
    full_df["SubClass"] = full_df["SubClass"].astype(str).str.strip()
    full_df["SubClass"] = full_df["SubClass"].replace("Flooding", "DoS")

    # 패킷 단위 문자열 라벨
    raw_labels = full_df["SubClass"].astype(str).values
    packet_labels_int = np.vectorize(LABEL_MAP.get)(raw_labels).astype(np.int64)

    # 4. 피처 계산에 필요한 컬럼 → numpy
    timestamps = full_df["Timestamp"].astype(np.float64).to_numpy()
    can_ids    = full_df["Arbitration_ID"].apply(parse_id).astype(np.int64).to_numpy()
    payload_array = np.vstack(
        full_df["Data"].apply(parse_payload_str).values
    ).astype(np.uint8)

    # 🔹 Normal 기반 Markov 자원 생성 (id_to_state, state_sequences, transition_ref_map)
    id_to_state, state_sequences, transition_ref_map = build_markov_resources(
        timestamps, can_ids, packet_labels_int
    )
    print("[INFO] 통합 데이터 피처 계산 중...")
    all_features = calculate_features_numba(
        timestamps, can_ids, state_sequences, payload_array, transition_ref_map
    )
    # 5. ===== 패킷 레벨 CSV 저장 =====
    packet_labels_int = np.vectorize(LABEL_MAP.get)(raw_labels).astype(np.int64)

    df_packet = pd.DataFrame(all_features, columns=FEATURE_NAMES)
    df_packet["Label_Int"] = packet_labels_int
    df_packet["Label_Str"] = raw_labels
    df_packet.to_csv(CSV_SAVE_PATH, index=False)
    print(f"[DONE] .csv 패킷 단위 저장 완료: {CSV_SAVE_PATH}")
    print(f"       형태: {df_packet.shape} (행: 패킷 수, 열: 특징+라벨)")

    # 6. ===== 윈도우 레벨 PT 저장 =====
    X_list = []
    y_list = []  # 이제 각 윈도우당 64개의 라벨을 저장

    num_packets = all_features.shape[0]
    for start in range(0, num_packets - WINDOW_SIZE + 1, STRIDE):
        end = start + WINDOW_SIZE
        
        # Feature: (64, 9) → (9, 64) 로 transpose
        X_list.append(all_features[start:end].T)
        
        # ✨ 핵심 변경: 윈도우 내 64개 패킷의 라벨을 모두 저장
        window_labels = packet_labels_int[start:end]  # (64,) 형태
        y_list.append(window_labels)

    X = np.array(X_list)     # (num_windows, 9, 64)
    y = np.array(y_list)     # (num_windows, 64)  ← 변경됨!

    np.savez(
    'C:/Users/user/Desktop/IDS_masters/dataset/carchallenge_training_0209(yoonju).npz',
    X=X.astype(np.float32),
    y=y.astype(np.int64)
    )
    
    # [추가] 일반화를 위한 매핑 정보 저장
    save_dir = "C:/Users/user/Desktop/IDS_masters/dataset/"
    os.makedirs(save_dir, exist_ok=True)

    # 1. ID -> State 매핑 사전 저장
    joblib.dump(id_to_state, os.path.join(save_dir, "id_to_state.pkl"))
    # 2. Markov 전이 확률 맵 저장
    joblib.dump(transition_ref_map, os.path.join(save_dir, "markov_ref_map.pkl"))

    print(f"✅ Markov Reference Maps saved to {save_dir}")

    print(f" Saved dataset")

if __name__ == "__main__":
    main()

[INFO] 발견된 파일: 2개
   - Pre_submit_D.csv
   - Pre_submit_S.csv
[READING] Pre_submit_D.csv 읽는 중...
[READING] Pre_submit_S.csv 읽는 중...
[INFO] 통합 완료. 총 패킷 수: 3752046
[INFO] 통합 데이터 피처 계산 중...
[DONE] .csv 패킷 단위 저장 완료: C:/Users/user/Desktop/IDS_masters/dataset/training_dataset4.csv
       형태: (3752046, 8) (행: 패킷 수, 열: 특징+라벨)
 Saved dataset
